# Connector: ticketers and the approval loop

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pecca-core/pecca/blob/main/examples/notebooks/connector_ticketer.ipynb)

A `Ticketer` opens an approval ticket when a promotion needs sign-off and reports who approved. We run the full loop (train → shadow → `live` blocked → ticket → approval → `live`) with **manual**, **GitHub Issues** and **Jira** approvals. GitHub and Jira HTTP is mocked.

In [1]:
%pip install -q pecca respx pyyaml

In [2]:
import json, os, re, shutil, subprocess, tempfile, time
from pathlib import Path

import pandas as pd
import pecca
from pecca.core import context
from pecca.data.dataset import Dataset, canonicalize
from pecca.data.synthetic import generate

HERE = Path.cwd()                                   # the notebook's folder (examples/notebooks in the repo)
work = tempfile.mkdtemp()
os.chdir(work)
os.environ["PECCA_HOME"] = os.path.join(work, ".pecca")
os.environ["TQDM_DISABLE"] = "1"
os.environ["MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT"] = "1"
import logging
try:
    import mlflow  # MLflow resets its logger level on import, so quiet it afterwards
    logging.getLogger("mlflow").setLevel(logging.ERROR)
except ImportError:
    pass

# Docker-backed cells run only when Docker is reachable and PECCA_SKIP_DOCKER != 1 (CI sets it).
USE_DOCKER = (os.environ.get("PECCA_SKIP_DOCKER") != "1" and shutil.which("docker") is not None
              and subprocess.run(["docker", "info"], capture_output=True).returncode == 0)
print("docker:", "available" if USE_DOCKER else "skipped (PECCA_SKIP_DOCKER=1 or Docker not reachable)")

def sh(*args, check=True):
    return subprocess.run([str(a) for a in args], capture_output=True, text=True, check=check)

import atexit
def cleanup_later(*cmd):
    """Run `cmd` when the kernel exits, so containers are removed even if a cell fails."""
    atexit.register(lambda: subprocess.run([str(c) for c in cmd], capture_output=True))

def free_port():
    import socket
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

def wait_for(check, what, seconds=90):
    end = time.time() + seconds
    while time.time() < end:
        try:
            if check():
                return
        except Exception:
            pass
        time.sleep(1)
    raise TimeoutError(f"{what} did not become ready in {seconds}s")

def train_demo(call="route_rfi", rows=3000, candidates=("tfidf_linear",), **kw):
    """Train a small model on the synthetic demo data (LLM answers + human corrections)."""
    df = generate(rows)
    df["text"] = df.email_subject + "\n\n" + df.email_body
    cols = {"input": "text", "llm_output": "llm_rfi", "human_label": "human_rfi", "call_name": {"literal": call.split("/")[-1]}}
    return pecca.train(call, Dataset(canonicalize(df, cols), cols), candidates=list(candidates), **kw)

docker: available


In [3]:
import httpx, respx, yaml

def project_cfg(name, ticketer, via):
    gov = {"extends": "none", "transitions": {"shadow->live": {"gates": ["shadow_days >= 0"],
           "approvals": {"groups": ["model-risk"], "require": "all", "via": via},
           "evidence": ["model_card", "threshold_rationale"]}}}
    proj = {"governance": gov}
    if ticketer:
        proj["integrations"] = {"ticketer": ticketer}
    return proj

def configure(**projects):
    Path("pecca.yaml").write_text(yaml.safe_dump({"version": 1, "projects": projects}))
    context.reset()

def show(dec):
    return {"allowed": dec.allowed, "pending": dec.pending_approvals, "evidence missing": dec.evidence_missing}

## Manual approvals
`via: manual`: someone runs `pecca approve` (or `pecca.approve`) and names the group they represent.

In [4]:
configure(manual=project_cfg("manual", {"type": "manual"}, "manual"))
train_demo("manual/route_rfi")
pecca.promote("manual/route_rfi", "shadow")
print("blocked :", show(pecca.promote("manual/route_rfi", "live")))
pecca.approve("manual/route_rfi", "v1", approver="mrm@bank.example", groups=["model-risk"], note="reviewed")
print("approved:", show(pecca.promote("manual/route_rfi", "live")))
print("mode now:", context.get_call("manual/route_rfi").state().mode)

blocked : {'allowed': False, 'pending': ['group:model-risk'], 'evidence missing': []}
approved: {'allowed': True, 'pending': [], 'evidence missing': []}
mode now: live


## GitHub Issues
Pecca opens an issue; an approval is a `/approve` comment by a member of the group (`group_members`).

In [5]:
configure(gh=project_cfg("gh", {"type": "github", "repo": "bank/model-risk", "token": "test-token", "group_members": {"model-risk": ["alice"]}}, "github:bank/model-risk"))
train_demo("gh/route_rfi")
pecca.promote("gh/route_rfi", "shadow")
with respx.mock:
    issue = respx.post("https://api.github.com/repos/bank/model-risk/issues").mock(return_value=httpx.Response(201, json={"number": 41}))
    thread = [{"body": "looks fine to me", "user": {"login": "bob"}}]          # bob is not in the model-risk group
    respx.get("https://api.github.com/repos/bank/model-risk/issues/41/comments").mock(side_effect=lambda request: httpx.Response(200, json=thread))
    print("1. blocked, ticket opened:", show(pecca.promote("gh/route_rfi", "live")))
    print("   ticket:", context.get_call("gh/route_rfi").state().tickets, "| issue title:", json.loads(issue.calls[0].request.content)["title"])
    print("2. bob commented only   :", show(pecca.promote("gh/route_rfi", "live")))
    thread.append({"body": "/approve", "user": {"login": "alice"}})              # alice is in model-risk
    print("3. alice: /approve      :", show(pecca.promote("gh/route_rfi", "live")))
print("mode now:", context.get_call("gh/route_rfi").state().mode)

1. blocked, ticket opened: {'allowed': False, 'pending': ['group:model-risk'], 'evidence missing': []}
   ticket: {'v1': '41'} | issue title: Pecca: approve default/gh/route_rfi v1 (shadow->live)
2. bob commented only   : {'allowed': False, 'pending': ['group:model-risk'], 'evidence missing': []}


3. alice: /approve      : {'allowed': True, 'pending': [], 'evidence missing': []}
mode now: live


## Jira
An approval is a status transition to `approved_when.status` (the changelog author is the approver).

In [6]:
configure(jira=project_cfg("jira", {"type": "jira", "url": "https://bank.atlassian.example", "project": "MRM", "email": "pecca@bank.example", "token": "test-token",
                                    "approved_when": {"status": "Approved"}, "group_members": {"model-risk": ["mrm@bank.example"]}}, "jira:MRM"))
train_demo("jira/route_rfi")
pecca.promote("jira/route_rfi", "shadow")
histories = [{"author": {"emailAddress": "dev@bank.example"}, "created": "2026-10-05", "items": [{"field": "status", "toString": "In Review"}]}]
with respx.mock:
    respx.post("https://bank.atlassian.example/rest/api/3/issue").mock(return_value=httpx.Response(201, json={"key": "MRM-7"}))
    respx.get(url__regex=r".*/issue/MRM-7\?expand=changelog.*").mock(side_effect=lambda request: httpx.Response(200, json={"changelog": {"histories": histories}}))
    print("1. blocked, ticket opened:", show(pecca.promote("jira/route_rfi", "live")))
    print("   ticket:", context.get_call("jira/route_rfi").state().tickets)
    print("2. moved to In Review   :", show(pecca.promote("jira/route_rfi", "live")))
    histories.append({"author": {"emailAddress": "mrm@bank.example"}, "created": "2026-10-06", "items": [{"field": "status", "toString": "Approved"}]})
    print("3. mrm@ set to Approved :", show(pecca.promote("jira/route_rfi", "live")))
print("mode now:", context.get_call("jira/route_rfi").state().mode)

1. blocked, ticket opened: {'allowed': False, 'pending': ['group:model-risk'], 'evidence missing': []}
   ticket: {'v1': 'MRM-7'}
2. moved to In Review   : {'allowed': False, 'pending': ['group:model-risk'], 'evidence missing': []}


3. mrm@ set to Approved : {'allowed': True, 'pending': [], 'evidence missing': []}
mode now: live
